# 04 — Model Evaluation & Comparison

Compare all models on the test set:
- MAE per score dimension
- Correlation with ground truth
- Consistency (same image 5x → variance)
- Latency and cost per inference

**Models:** GPT-4o zero-shot, GPT-4o fine-tuned, MedGemma fine-tuned, Claude Vision (optional)

In [ ]:
!pip install -q openai pillow scikit-learn matplotlib pandas numpy tqdm scipy

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json, time, base64, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from openai import OpenAI
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

DATA_DIR = '/content/drive/MyDrive/RadianceIQ/datasets/processed'
MODELS_DIR = '/content/drive/MyDrive/RadianceIQ/models'
RESULTS_DIR = '/content/drive/MyDrive/RadianceIQ/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

with open(f'{DATA_DIR}/test.json') as f:
    test_data = json.load(f)
print(f'Test set: {len(test_data)} images')

## 1. Evaluation Helpers

In [ ]:
SYSTEM_PROMPT = 'You are a dermatology analysis assistant. Analyze the skin photo and return JSON: {"acne_score": 0-100, "sun_damage_score": 0-100, "skin_age_score": 0-100, "confidence": "low"|"med"|"high", "primary_driver": str, "recommended_action": str}. Return ONLY valid JSON.'

def image_to_base64(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def parse_output(text):
    match = re.search(r'\{[\s\S]*?\}', text)
    if match:
        try: return json.loads(match.group())
        except: pass
    return None

def evaluate(preds, gt):
    metrics = {}
    for dim in ['acne_score', 'sun_damage_score', 'skin_age_score']:
        pred_vals = [p[dim] for p in preds if dim in p]
        true_vals = [g[dim] for g, p in zip(gt, preds) if dim in p]
        if len(pred_vals) < 2: continue
        mae = mean_absolute_error(true_vals, pred_vals)
        corr, p_val = pearsonr(true_vals, pred_vals)
        metrics[dim] = {'mae': round(mae, 2), 'correlation': round(corr, 3)}
    return metrics

def run_model(model_id, items, desc):
    preds, latencies, gt = [], [], []
    for item in tqdm(items, desc=desc):
        try:
            img_b64 = image_to_base64(item['image_path'])
            start = time.time()
            resp = client.chat.completions.create(
                model=model_id,
                messages=[{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': [{'type': 'text', 'text': 'Analyze this skin photo.'}, {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{img_b64}'}}]}],
                max_tokens=300, temperature=0.2,
            )
            result = parse_output(resp.choices[0].message.content)
            if result:
                preds.append(result)
                latencies.append(time.time() - start)
                gt.append(item)
        except Exception as e:
            print(f'Error: {e}')
    return preds, latencies, gt

## 2. Run Evaluations

In [ ]:
MAX_EVAL = 50

# GPT-4o zero-shot
zs_preds, zs_lat, zs_gt = run_model('gpt-4o', test_data[:MAX_EVAL], 'GPT-4o zero-shot')
zs_metrics = evaluate(zs_preds, zs_gt)
print(f'Zero-shot: {json.dumps(zs_metrics, indent=2)}, Avg latency: {np.mean(zs_lat):.2f}s')

# GPT-4o fine-tuned
with open(f'{MODELS_DIR}/gpt4o/model_info.json') as f:
    ft_model_id = json.load(f)['model_id']

ft_preds, ft_lat, ft_gt = run_model(ft_model_id, test_data[:MAX_EVAL], 'GPT-4o fine-tuned')
ft_metrics = evaluate(ft_preds, ft_gt)
print(f'Fine-tuned: {json.dumps(ft_metrics, indent=2)}, Avg latency: {np.mean(ft_lat):.2f}s')

## 3. Consistency Test

In [ ]:
def consistency_test(model_id, image_path, n=5):
    img_b64 = image_to_base64(image_path)
    results = []
    for _ in range(n):
        try:
            resp = client.chat.completions.create(
                model=model_id,
                messages=[{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': [{'type': 'text', 'text': 'Analyze.'}, {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{img_b64}'}}]}],
                max_tokens=300, temperature=0.2,
            )
            r = parse_output(resp.choices[0].message.content)
            if r: results.append(r)
        except: pass
    if len(results) < 2: return None
    return {dim: round(np.var([r[dim] for r in results if dim in r]), 2) for dim in ['acne_score', 'sun_damage_score', 'skin_age_score']}

consistency = {}
for name, mid in [('GPT-4o Zero-shot', 'gpt-4o'), ('GPT-4o Fine-tuned', ft_model_id)]:
    vars_list = [consistency_test(mid, item['image_path']) for item in test_data[:3]]
    vars_list = [v for v in vars_list if v]
    consistency[name] = {dim: round(np.mean([v[dim] for v in vars_list]), 2) for dim in ['acne_score', 'sun_damage_score', 'skin_age_score']}

print('Consistency (avg variance):', json.dumps(consistency, indent=2))

## 4. Charts

In [ ]:
models = ['GPT-4o Zero-shot', 'GPT-4o Fine-tuned']
all_metrics = [zs_metrics, ft_metrics]
all_latencies = [np.mean(zs_lat), np.mean(ft_lat)]
dims = ['acne_score', 'sun_damage_score', 'skin_age_score']
labels = ['Acne', 'Sun Damage', 'Skin Age']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (dim, label) in enumerate(zip(dims, labels)):
    maes = [m.get(dim, {}).get('mae', 0) for m in all_metrics]
    bars = axes[i].bar(models, maes, color=['#4CAF50', '#2196F3'])
    axes[i].set_title(f'{label} MAE')
    axes[i].set_ylabel('Mean Absolute Error')
    for bar, val in zip(bars, maes):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}', ha='center')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/mae_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Generate Report

In [ ]:
report = '# RadianceIQ Model Evaluation Report\n\n'
report += '## MAE per Score Dimension\n\n| Model | Acne | Sun Damage | Skin Age |\n|-------|------|------------|----------|\n'
for name, m in zip(models, all_metrics):
    report += f'| {name} | {m.get("acne_score", {}).get("mae", "N/A")} | {m.get("sun_damage_score", {}).get("mae", "N/A")} | {m.get("skin_age_score", {}).get("mae", "N/A")} |\n'

report += '\n## Correlation\n\n| Model | Acne r | Sun Damage r | Skin Age r |\n|-------|--------|-------------|-----------|\n'
for name, m in zip(models, all_metrics):
    report += f'| {name} | {m.get("acne_score", {}).get("correlation", "N/A")} | {m.get("sun_damage_score", {}).get("correlation", "N/A")} | {m.get("skin_age_score", {}).get("correlation", "N/A")} |\n'

report += f'\n## Latency\n\n| Model | Avg (s) |\n|-------|---------|\n'
for name, lat in zip(models, all_latencies):
    report += f'| {name} | {lat:.2f} |\n'

report += '\n## Consistency (Avg Variance)\n\n| Model | Acne | Sun Damage | Skin Age |\n|-------|------|------------|----------|\n'
for name in models:
    if name in consistency:
        c = consistency[name]
        report += f'| {name} | {c["acne_score"]} | {c["sun_damage_score"]} | {c["skin_age_score"]} |\n'

report += '\n## Recommendation\n\nSelect the model with lowest MAE, highest correlation, lowest variance, and acceptable latency/cost.\n'

with open(f'{RESULTS_DIR}/evaluation_report.md', 'w') as f:
    f.write(report)
print(report)